# Приложение А к Модулю 11.5. Лесоруб уезжает на HF Spaces

В основном ноутбуке домашки вы собрали хороший набор tools — `move`, `gather`, `deposit`, `get_map` — и в Блоке 3 живая модель уже решала им задачу лесоруба. Здесь делаем следующий шаг: превращаем тот же набор в веб-чат на Hugging Face Spaces по механике [Модуля 10.7 «Деплой агента»](https://itrubnikov.github.io/Train_of_Thought/docs/modules/10-7-deploy-agent/) — три файла (`app.py`, `requirements.txt`, `README.md`), и агент живёт по ссылке, которой можно делиться.

План простой:

- собираем `app.py`: код `Forest` и финальных инструментов — дословно из основного ноутбука, плюс пять строк чата;
- рядом кладём `requirements.txt` и `README.md` с YAML-шапкой;
- прогоняем смоук без сети: импортируем `app.py` и дёргаем tools руками;
- деплой за три клика — и задача лесоруба уходит агенту прямо в чате.

Главное про запуск: для прогона этого ноутбука не нужно **ничего** — ни токена, ни сети (после установки пакета в первой ячейке). `HF_TOKEN` понадобится только в Secrets самого Space, когда понесёте файлы на huggingface.co. Эталонные копии всех трёх файлов лежат в репозитории: `spaces/module-11-5-agent/`.

In [ ]:
import sys, subprocess

def _pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                   check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

try:
    import smolagents  # noqa: F401
except Exception:
    _pip_install("smolagents")
    import smolagents  # noqa: F401

try:
    import pydantic  # noqa: F401
except Exception:
    _pip_install("pydantic")
    import pydantic  # noqa: F401

print("smolagents", smolagents.__version__, "| pydantic", pydantic.VERSION)

## Шаг 1. Собираем `app.py`

Весь код вам уже знаком: класс `Forest` с `reset_forest` — из Блока 1, `get_map` — из шага 1 арки, финальные `MoveTool` / `GatherTool` / `deposit` — из шагов 4–5, с конвертом `{result, cooldown, state}` и обучающими ошибками. Копируем дословно, без «улучшений»: контракт уже вылизан домашкой, а менять его при деплое — лучший способ что-нибудь сломать.

Новых строк ровно пять — блок в самом конце файла:

```python
if __name__ == "__main__":
    from smolagents import InferenceClientModel, ToolCallingAgent, GradioUI
    model = InferenceClientModel()   # HF_TOKEN возьмёт из Secrets Space
    agent = ToolCallingAgent(tools=[move, gather, deposit, get_map], model=model, max_steps=12)
    GradioUI(agent).launch()
```

Зачем `if __name__ == "__main__"`: на Spaces `app.py` запускается как главный скрипт — `__name__` равно `"__main__"`, и чат стартует. А когда мы **импортируем** файл (так сделает смоук ниже), `__name__` другой — блок молчит, ни модель, ни сеть не трогаются. Один файл, два режима.

Ячейка ниже держит полный текст `app.py` в переменной `APP_PY` и записывает его в папку `./space-files/`.

In [ ]:
import pathlib

APP_PY = '''"""Приложение А к Модулю 11.5 — лесоруб уезжает на HF Spaces.

Хороший набор tools из домашки (move, gather, deposit, get_map), отданный живой
модели через ToolCallingAgent и обёрнутый в веб-чат GradioUI — по механике
Модуля 10.7 «Деплой агента». Классы Forest и все инструменты скопированы из
notebooks/module-11-5-tool-design/notebook.ipynb без изменений.

Для работы чата на HF Spaces нужен секрет HF_TOKEN
(Space -> Settings -> Variables and secrets). При импорте файла как модуля
сеть не трогается: живой блок внизу спрятан под if __name__ == "__main__".
"""
from smolagents import tool, Tool

import time


class Forest:
    """Мок-лес: сетка 5x5, лесоруб, рюкзак, склад и кулдаун."""

    COOLDOWN = 0.3  # в лекции 1.0; здесь меньше, чтобы Run all занимал ~минуту, — правила те же

    def __init__(self):
        self.size = 5
        self.nodes = {(1, 2): "wood", (3, 1): "wood", (2, 4): "stone"}
        self.pos = (0, 0)        # где стоит лесоруб
        self.home = (0, 0)       # клетка склада
        self.backpack = {}       # например, {"wood": 3}
        self.cap = 5             # вместимость рюкзака
        self.stock = {}          # что уже сдано на склад
        self.busy_until = 0.0    # когда закончится кулдаун

    def cooldown_left(self):
        return max(0.0, self.busy_until - time.monotonic())

    def start_cooldown(self):
        self.busy_until = time.monotonic() + self.COOLDOWN

    def backpack_load(self):
        return sum(self.backpack.values())


forest = Forest()


def reset_forest():
    """Свежий мир перед каждым сценарием. Tools ниже смотрят на глобальную forest."""
    global forest
    forest = Forest()


@tool
def get_map() -> dict:
    """Карта леса: позиция лесоруба, узлы ресурсов и клетка склада. Мир не меняет."""
    return {"pos": list(forest.pos), "home": list(forest.home),
            "nodes": [{"pos": list(p), "resource": r} for p, r in forest.nodes.items()]}


class GatherTool(Tool):
    name = "gather"
    description = (
        "Добыть один ресурс с клетки, на которой стоит лесоруб. "
        "Без resource берёт то, что есть на клетке; с resource — только ожидаемое."
    )
    inputs = {
        "resource": {
            "type": "string",
            "enum": ["wood", "stone"],
            "nullable": True,
            "description": "Ожидаемый ресурс: wood или stone. По умолчанию — любой.",
        }
    }
    output_type = "object"

    def forward(self, resource: str | None = None) -> dict:
        wait = forest.cooldown_left()
        if wait > 0:
            return {"error": {"code": "on_cooldown",
                              "message": f"Лесоруб занят ещё {wait:.1f} с — подождите и повторите."}}
        node = forest.nodes.get(forest.pos)
        if node is None or (resource is not None and node != resource):
            return {"error": {"code": "no_resource_here",
                              "message": "На этой клетке нет нужного узла — "
                                         "найдите его через get_map и подойдите move."}}
        if forest.backpack_load() >= forest.cap:
            return {"error": {"code": "inventory_full",
                              "message": f"Рюкзак полон ({forest.cap}/{forest.cap}) — "
                                         "вернитесь на склад (0, 0) и позовите deposit."}}
        forest.backpack[node] = forest.backpack.get(node, 0) + 1
        forest.start_cooldown()
        return {"result": {"gathered": node, "amount": 1},
                "cooldown": Forest.COOLDOWN,
                "state": {"pos": list(forest.pos),
                          "backpack": dict(forest.backpack), "cap": forest.cap}}


class MoveTool(Tool):
    name = "move"
    description = (
        "Шаг на одну клетку в сторону direction. "
        "Зовите, когда до нужного узла или склада не хватает шага."
    )
    inputs = {
        "direction": {
            "type": "string",
            "enum": ["north", "south", "east", "west"],
            "description": "Куда шагнуть: north, south, east или west.",
        }
    }
    output_type = "object"

    def forward(self, direction: str) -> dict:
        wait = forest.cooldown_left()
        if wait > 0:
            return {"error": {"code": "on_cooldown",
                              "message": f"Лесоруб занят ещё {wait:.1f} с — подождите и повторите."}}
        dx, dy = {"north": (0, -1), "south": (0, 1),
                  "east": (1, 0), "west": (-1, 0)}[direction]
        x, y = forest.pos
        forest.pos = (min(max(x + dx, 0), forest.size - 1),
                      min(max(y + dy, 0), forest.size - 1))
        forest.start_cooldown()
        return {"result": {"pos": list(forest.pos)},
                "cooldown": Forest.COOLDOWN,
                "state": {"pos": list(forest.pos),
                          "backpack": dict(forest.backpack), "cap": forest.cap}}


@tool
def deposit() -> dict:
    """Сдать содержимое рюкзака на склад. Работает только на клетке склада (0, 0)."""
    wait = forest.cooldown_left()
    if wait > 0:
        return {"error": {"code": "on_cooldown",
                          "message": f"Лесоруб занят ещё {wait:.1f} с — подождите и повторите."}}
    if forest.pos != forest.home:
        return {"error": {"code": "not_at_storehouse",
                          "message": f"Склад на клетке (0, 0), а вы — на {forest.pos}. "
                                     "Дойдите до склада и повторите deposit."}}
    banked, forest.backpack = forest.backpack, {}
    for res, n in banked.items():
        forest.stock[res] = forest.stock.get(res, 0) + n
    forest.start_cooldown()
    return {"result": {"banked": banked},
            "cooldown": forest.COOLDOWN,
            "state": {"pos": list(forest.pos), "backpack": {}, "stock": forest.stock}}


move, gather = MoveTool(), GatherTool()
if __name__ == "__main__":
    # На Spaces app.py запускается как главный скрипт -> чат стартует сам.
    # При импорте (смоук в ноутбуке-приложении) __name__ другой -> блок молчит.
    from smolagents import InferenceClientModel, ToolCallingAgent, GradioUI

    model = InferenceClientModel()   # HF_TOKEN возьмёт из Secrets Space
    agent = ToolCallingAgent(tools=[move, gather, deposit, get_map], model=model, max_steps=12)
    GradioUI(agent).launch()
'''

space_dir = pathlib.Path("space-files")
space_dir.mkdir(exist_ok=True)
(space_dir / "app.py").write_text(APP_PY, encoding="utf-8")
print("Записан", space_dir / "app.py", "|", len(APP_PY.splitlines()), "строк")
print("Внутри: Forest + reset_forest, tools move/gather/deposit/get_map,")
print("живой блок — только под if __name__ == '__main__'.")

## Шаг 2. Два соседних файла

`requirements.txt` — две строки: `smolagents` и `gradio>=5,<6`. `README.md` — не просто описание: его YAML-шапка — это метаданные сборки Space (`sdk: gradio`, `sdk_version`, `app_file`). Записываем оба файла и печатаем содержимое.

In [ ]:
REQUIREMENTS_TXT = "smolagents\ngradio>=5,<6\n"

README_MD = '''---
title: Lumberjack Agent 11.5
emoji: 🪓
colorFrom: green
colorTo: yellow
sdk: gradio
sdk_version: 5.49.1
app_file: app.py
pinned: false
short_description: Лесоруб из Модуля 11.5 — хорошие tools в чате на HF Spaces
---

# Лесоруб — Приложение А к Модулю 11.5

Готовая HF Space-заготовка к домашке [Модуля 11.5: Дизайн инструментов](https://itrubnikov.github.io/Train_of_Thought/docs/modules/11-5-tool-design/). Тот же хороший набор tools, что вы собрали в ноутбуке, — `move`, `gather`, `deposit`, `get_map` с `enum` в схеме, обучающими ошибками `{code, message}` и конвертом `{result, cooldown, state}` — отдан живой модели через `ToolCallingAgent` и обёрнут в чат `GradioUI` по механике [Модуля 10.7: Деплой агента](https://itrubnikov.github.io/Train_of_Thought/docs/modules/10-7-deploy-agent/). Напишите агенту в чате: «Добудь одно дерево и сдай его на склад» — и смотрите, как контракт ведёт модель к цели.

## Как развернуть

### Вариант A — создать Space из этих файлов
1. huggingface.co → **New → Space**, SDK — **Gradio**.
2. Загрузите `app.py`, `requirements.txt`, `README.md` (через *Files → Add file* или `git push` в репозиторий Space).
3. **Settings → Variables and secrets → New secret**: `HF_TOKEN` = бесплатный токен (huggingface.co → Settings → Access Tokens, роль `read`).
4. Space соберётся сам → откройте чат и дайте агенту задачу лесоруба.

### Вариант B — локально
```bash
pip install -r requirements.txt
export HF_TOKEN=hf_...
python app.py     # откроется http://127.0.0.1:7860
```

## Модель
`app.py` вызывает `InferenceClientModel()` без аргументов — в smolagents 1.26.0 это `Qwen/Qwen3-Next-80B-A3B-Thinking`: умная, но заметно дороже по кредитам, чем 7B-coder из Модуля 10.7. Хотите экономнее, модель недоступна или кончились кредиты — впишите другую из [hf.co/models?inference=warm](https://huggingface.co/models?inference=warm): `InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-7B-Instruct")`.

## Подводные камни
- **`sdk_version` только 5.x.** gradio 4.x падает на Python 3.13 (`ModuleNotFoundError: audioop`), на котором HF собирает Spaces.
- **Нет секрета `HF_TOKEN` → агент молчит.** Токен только в секрете, никогда в `app.py`.
- **Бесплатный Space засыпает** — первый ответ после простоя идёт дольше; каждый вызов агента тратит бесплатные кредиты HF Inference.
- **`max_steps=12` в `app.py` бережёт квоту**: даже заблудившийся агент не сделает больше 12 шагов за задачу.
- **Лес общий на процесс.** `Forest` — глобальный объект: все, кто откроет чат, играют в одном лесу. Для учебного демо это нормально.
'''

(space_dir / "requirements.txt").write_text(REQUIREMENTS_TXT, encoding="utf-8")
(space_dir / "README.md").write_text(README_MD, encoding="utf-8")

print("--- space-files/requirements.txt " + "-" * 27)
print(REQUIREMENTS_TXT)
print("--- space-files/README.md " + "-" * 34)
print(README_MD)

## Шаг 3. Смоук без сети

Прежде чем нести файлы на huggingface.co — проверим их локально. Импортируем `./space-files/app.py` как модуль через `importlib`. Почему именно импорт:

- `exec()` строки не подойдёт: `@tool` читает исходник функции через `inspect.getsource`, а у кода внутри `exec` файла-исходника нет — упадёт (грабли из README основного ноутбука);
- при импорте `__name__` модуля — не `"__main__"`, поэтому живой блок с моделью и `launch()` не выполнится: смоук честно офлайновый.

Дальше знакомый маршрут: конверт `{result, cooldown, state}` на шаге, обучающая ошибка на пустой клетке и полный мини-рейс — до дерева, добыть, вернуться на склад, сдать.

In [ ]:
import importlib.util
import json
import sys
import time

spec = importlib.util.spec_from_file_location("space_app", space_dir / "app.py")
space_app = importlib.util.module_from_spec(spec)
sys.modules["space_app"] = space_app
spec.loader.exec_module(space_app)   # __name__ == "space_app" -> живой блок не выполнился
print("Импорт прошёл: модуль", space_app.__name__, "| tools:",
      space_app.move.name, space_app.gather.name,
      space_app.deposit.name, space_app.get_map.name)


def wait_ready():
    time.sleep(space_app.forest.cooldown_left() + 0.02)


space_app.reset_forest()

# 1. Конверт в деле: шаг на восток
step = space_app.move(direction="east")
print()
print("move(direction='east') ->", json.dumps(step, ensure_ascii=False))
assert step["result"]["pos"] == [1, 0]
assert "cooldown" in step and "state" in step

# 2. Обучающая ошибка: добыть на пустой клетке
wait_ready()
miss = space_app.gather()
print("gather() на пустой клетке ->", json.dumps(miss, ensure_ascii=False))
assert miss["error"]["code"] == "no_resource_here"

# 3. Мини-рейс: дойти до дерева (1, 2), добыть, вернуться на склад, сдать
for d in ("south", "south"):
    wait_ready()
    space_app.move(direction=d)
wait_ready()
wood = space_app.gather(resource="wood")
assert wood["result"] == {"gathered": "wood", "amount": 1}
for d in ("north", "north", "west"):
    wait_ready()
    space_app.move(direction=d)
wait_ready()
banked = space_app.deposit()
print("deposit() ->", json.dumps(banked, ensure_ascii=False))
assert banked["result"]["banked"] == {"wood": 1}
assert space_app.forest.stock == {"wood": 1}

print()
print("Смоук зелёный: app.py импортируется, конверт и обучающие ошибки на месте, сеть не тронута.")

## Деплой за три клика

1. На huggingface.co: **New → Space**, SDK — **Gradio**.
2. Залейте три файла из `./space-files/` (или эталонные из репозитория, папка `spaces/module-11-5-agent/`) — через *Files → Add file* или `git push` в репозиторий Space.
3. **Settings → Variables and secrets → New secret**: `HF_TOKEN` = ваш бесплатный токен (тот же, что в Блоке 3 основного ноутбука).
4. Space соберётся сам → откройте URL → напишите агенту: «Добудь одно дерево и сдай его на склад».

Грабли (те же, что в Модуле 10.7):

- `sdk_version` в README-шапке — обязательно **5.x**: gradio 4.x падает на Python 3.13 (`ModuleNotFoundError: audioop`), на котором HF собирает Spaces;
- без секрета `HF_TOKEN` агент молчит — ключ всегда секретом, никогда в `app.py`;
- бесплатный Space засыпает — первый ответ после простоя идёт дольше, а каждый вызов агента тратит бесплатные кредиты HF Inference;
- `max_steps=12` в `app.py` — не украшение, а страховка квоты: даже заблудившийся агент не сделает больше 12 шагов за задачу.

## Что дальше

- **Другой движок — одна строка.** Замените в `app.py` `ToolCallingAgent` на `CodeAgent` — агент станет писать код вместо JSON-вызовов, а контракт tools не изменится ни на символ. Помните только, что `CodeAgent` исполняет сгенерированный моделью код: для недоверенных входов нужен sandbox (Модуль 9.5).
- **Эталонные файлы** лежат в репозитории: `spaces/module-11-5-agent/` — можно деплоить прямо их.
- **Тот же контракт против реальности** — Блок 4 основного ноутбука: живой API игры отвечает теми же конвертами и теми же обучающими ошибками. Дизайн tools, который вы только что задеплоили, — боевой, не учебная поделка.